# jobs: balls64 (amort)

project = ```iP-VAE```, host = ```chewie```, device = ```any```

**Motivation**: <br>

Create jobs for BALLS64 dataset amortized VAE fits.

In [1]:
# HIDE CODE


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-progress-2025/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-progress-2025/figs')
tmp_dir = os.path.join(git_dir, 'jb-progress-2025/tmp')
git_dir = os.path.join(git_dir, 'PoissonVAE')

# GitHub
sys.path.insert(0, git_dir)
from figures.fighelper import *
from main.train_vae import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

## Setup

In [2]:
from analysis.helper import job_runner_script


def divide_list(lst: list, n: int):
	k, m = divmod(len(lst), n)
	lst_divided = [
		lst[
			i * k + min(i, m):
			(i + 1) * k + min(i + 1, m)
		] for i in range(n)
	]
	return lst_divided


def _cleanup(path, host=None):
    for f in os.listdir(path):
        cond = f.endswith('.txt')
        if host is not None:
            cond = cond and host in f
        if cond:
            os.remove(pjoin(path, f))


def _name(host, gpu_i, fit_i):
    return f"{host}-cuda{gpu_i}-fit{fit_i}"

In [3]:
save_dir = pjoin(git_dir, 'scripts')
os.makedirs(save_dir, exist_ok=True)

# delete existing job runners?
_cleanup(save_dir, None)

print(sorted(os.listdir(save_dir)))

['copyfits.sh', 'fit_vae.sh', 'kill_screens.sh', 'resume_fit.sh', 'run_sessions.sh']

## chewie (```amort```)

```<conv+b|lin>```

In [4]:
host = 'chewie'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)

In [5]:
model_act_map = {
    'poisson': [None],
    'gaussian': [None, 'relu'],
}
betas = [0.5, 1.0, 2.0, 5.0]
n_latents = 96
dataset = 'BALLS64'

In [6]:
tot = 0

for model_type, latent_act_list in model_act_map.items():
    for latent_act in latent_act_list:
        for beta in betas:
            arg = ' '.join(filter(None, [
                f"--kl_beta {beta}",
                f"--latent_act '{latent_act}'" if latent_act else '',
                f"--comment amort",
            ]))
            gpu_i = tot % torch.cuda.device_count()
            kws = dict(
                device=gpu_i,
                dataset='BALLS64',
                archi='conv+b|lin',
                model=model_type,
                args=arg,
                seed=1,
            )
            scripts[gpu_i].append(job_runner_script(**kws))
            tot += 1

In [7]:
print(tot)

12

In [8]:
scripts = dict(scripts)
print({k: len(v) for k, v in scripts.items()})

{0: 6, 1: 6}

### Save

In [9]:
n_fits = 2

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        print(combined.replace('&& ', '&& \n'))

[PROGRESS] 'chewie-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '0' 'BALLS64' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.5 --comment amort && 
./fit_vae.sh '0' 'BALLS64' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 2.0 --comment amort && 
./fit_vae.sh '0' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.5 --comment amort

[PROGRESS] 'chewie-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '0' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 2.0 --comment amort && 
./fit_vae.sh '0' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.5 --latent_act 'relu' --comment amort && 
./fit_vae.sh '0' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 2.0 --latent_act 'relu' --comment amort

[PROGRESS] 'chewie-cuda1-fit0.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '1' 'BALLS64' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 1.0 --comment amort && 
./fit_vae.sh '1' 'BALLS64' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 5.0 --comment amort && 
./fit_vae.sh '1' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.0 --comment amort

[PROGRESS] 'chewie-cuda1-fit1.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '1' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 5.0 --comment amort && 
./fit_vae.sh '1' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.0 --latent_act 'relu' --comment amort && 
./fit_vae.sh '1' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 5.0 --latent_act 'relu' --comment amort

In [10]:
"""n_fits = 3

for gpu_i, scripts in scripts_mach.items():
    scripts_divided = divide_list(scripts, n_fits)
    for fit_i, strings_list in enumerate(scripts_divided):
        # sort so BALLS64 don't conicide together
        sorted_strings = sorted(
            strings_list,
            key=lambda x: "BALLS64" in x,
            reverse=fit_i % 2 == 0,
        )
        combined = ' && '.join(sorted_strings)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        print(combined.replace('&& ', '&& \n'))"""

'n_fits = 3\n\nfor gpu_i, scripts in scripts_mach.items():\n    scripts_divided = divide_list(scripts, n_fits)\n    for fit_i, strings_list in enumerate(scripts_divided):\n        # sort so BALLS64 don\'t conicide together\n        sorted_strings = sorted(\n            strings_list,\n            key=lambda x: "BALLS64" in x,\n            reverse=fit_i % 2 == 0,\n        )\n        combined = \' && \'.join(sorted_strings)\n        save_obj(\n            obj=combined,\n            file_name=_name(host, gpu_i, fit_i),\n            save_dir=save_dir,\n            mode=\'txt\',\n        )\n        print(combined.replace(\'&& \', \'&& \n\'))'

Print one to check

In [11]:
print(combined.replace('&& ', '&& \n'))

./fit_vae.sh '1' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 5.0 --comment amort && 
./fit_vae.sh '1' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.0 --latent_act 'relu' --comment amort && 
./fit_vae.sh '1' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 5.0 --latent_act 'relu' --comment amort

In [12]:
print(scripts)

{
    0: [
        "./fit_vae.sh '0' 'BALLS64' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.5 --comment amort",
        "./fit_vae.sh '0' 'BALLS64' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 2.0 --comment amort",
        "./fit_vae.sh '0' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.5 --comment amort",
        "./fit_vae.sh '0' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 2.0 --comment amort",
        "./fit_vae.sh '0' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.5 --latent_act 'relu' --comment 
amort",
        "./fit_vae.sh '0' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 2.0 --latent_act 'relu' --comment 
amort"
    ],
    1: [
        "./fit_vae.sh '1' 'BALLS64' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 1.0 --comment amort",
        "./fit_vae.sh '1' 'BALLS64' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 5.0 --comment amort",
        "./fit_vae.sh '1' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.0 --comment amort",
        "./fit_vae.sh '1' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 5.0 --comment amort",
        "./fit_vae.sh '1' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.0 --latent_act 'relu' --comment 
amort",
        "./fit_vae.sh '1' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 5.0 --latent_act 'relu' --comment 
amort"
    ]
}

In [13]:
print(scripts_divided)

[
    [
        "./fit_vae.sh '1' 'BALLS64' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 1.0 --comment amort",
        "./fit_vae.sh '1' 'BALLS64' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 5.0 --comment amort",
        "./fit_vae.sh '1' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.0 --comment amort"
    ],
    [
        "./fit_vae.sh '1' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 5.0 --comment amort",
        "./fit_vae.sh '1' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.0 --latent_act 'relu' --comment 
amort",
        "./fit_vae.sh '1' 'BALLS64' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 5.0 --latent_act 'relu' --comment 
amort"
    ]
]